# 📈 Tech Challenge — Fase 02
## Previsão de Tendência do IBOVESPA com Dados Externos

**Curso:** Pós-graduação em Data Analytics — POSTECH  
**Autora:** Ana Raquel  
**Período dos dados:** Janeiro/2020 a Março/2026

---

## 🎯 Objetivo

Desenvolver um modelo preditivo capaz de classificar se o índice **IBOVESPA** vai fechar em **alta (↑)** ou **baixa (↓)** no dia seguinte, com acurácia mínima de **75%** no conjunto de teste (últimos 30 dias).

---

## 🧭 Contexto e Evolução do Projeto

Este notebook é a **terceira e mais completa versão** do projeto. As versões anteriores revelaram um problema fundamental: o histórico de preços do próprio IBOVESPA tem **sinal preditivo muito baixo** para a direção diária. Ampliar o período de dados de 2 para 6 anos não trouxe ganho — o melhor resultado com dados apenas do índice foi de 53,33%.

| Versão | Arquivo | Período | Features | Melhor acurácia | Modelo vencedor |
|--------|---------|---------|----------|-----------------|-----------------|
| V1 | `IBOVESPA_historicos_012024_032026.csv` | Jan/2024 → Mar/2026 | 12 | 60,00% | Random Forest |
| V2 | `IBOVESPA_historicos_012020_032026.csv` | Jan/2020 → Mar/2026 | 12 | 53,33% | Random Forest |
| **V3** | **`IBOVESPA_historicos_012020_032026.csv` + externos** | **Jan/2020 → Mar/2026** | **60+** | **?** | **?** |

**Hipótese desta versão:** dados de mercados externos (S&P500, VIX, dólar e petróleo) carregam informação que o IBOVESPA sozinho não consegue capturar, pois refletem forças macro globais que movimentam o mercado brasileiro.

---

## 📋 Estrutura do Notebook

1. Instalação e importação de bibliotecas
2. Carregamento do IBOVESPA (CSV local)
3. Download dos dados externos via yfinance
4. Fusão e alinhamento das bases
5. Análise Exploratória — correlações com o target
6. Engenharia de Atributos (IBOVESPA + externos)
7. Divisão treino/teste e validação cruzada temporal
8. Modelo 1 — Regressão Logística
9. Modelo 2 — Random Forest
10. Modelo 3 — Gradient Boosting
11. Comparação, Ensemble e escolha final
12. Justificativa técnica
13. Conclusão

---
## 1. 📦 Instalação e Importação de Bibliotecas

Esta versão utiliza a biblioteca **yfinance** para baixar dados financeiros diretamente do Yahoo Finance.
É necessário instalá-la antes de rodar o notebook — basta executar a célula abaixo uma única vez.

In [ ]:
# Instala o yfinance automaticamente no ambiente Python atual
# sys.executable garante que instala no mesmo Python do Jupyter, evitando conflitos
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', 'yfinance', '-q'], check=True)
print('✅ yfinance instalado!')

In [ ]:
# ── Manipulação e análise de dados ───────────────────────────────────────────
import pandas as pd                   # estrutura de dados e manipulação de tabelas
import numpy as np                    # operações numéricas e vetoriais
import yfinance as yf                 # download de dados financeiros (Yahoo Finance)

# ── Visualização ──────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt       # gráficos base
import matplotlib.dates as mdates     # formatação de datas nos eixos
import seaborn as sns                 # gráficos estatísticos (heatmap, etc.)

# ── Pré-processamento ─────────────────────────────────────────────────────────
from sklearn.preprocessing import StandardScaler  # normalização de features

# ── Modelos de Machine Learning ───────────────────────────────────────────────
from sklearn.linear_model import LogisticRegression          # modelo baseline linear
from sklearn.ensemble import RandomForestClassifier          # ensemble de árvores (paralelo)
from sklearn.ensemble import GradientBoostingClassifier      # ensemble boosting (sequencial)

# ── Métricas de avaliação ─────────────────────────────────────────────────────
from sklearn.metrics import (
    accuracy_score,          # % de acertos geral
    classification_report,   # precision, recall e F1 por classe
    confusion_matrix,        # matriz de acertos e erros por classe
    ConfusionMatrixDisplay   # visualização da matriz de confusão
)

# ── Validação temporal ────────────────────────────────────────────────────────
from sklearn.model_selection import TimeSeriesSplit  # cross-validation respeitando ordem temporal
from sklearn.model_selection import cross_val_score  # executa o CV e retorna scores

# ── Configurações visuais globais ─────────────────────────────────────────────
sns.set_theme(style='whitegrid', palette='muted')   # tema padrão dos gráficos
plt.rcParams['figure.figsize'] = (14, 5)            # tamanho padrão das figuras
plt.rcParams['axes.titlesize'] = 14                 # tamanho dos títulos

import warnings
warnings.filterwarnings('ignore')  # suprime alertas de versão e convergência

print('✅ Bibliotecas importadas com sucesso!')

---
## 2. 📥 Carregamento do IBOVESPA (CSV local)

O arquivo `IBOVESPA_historicos_012020_032026.csv` foi exportado manualmente do site [investing.com](https://br.investing.com/indices/bovespa-historical-data), cobrindo **6 anos de pregões diários** (Janeiro/2020 a Março/2026).

Esta base ampliada inclui eventos extremos como o **crash da COVID-19 (março/2020)**, a **recuperação pós-pandemia**, o **ciclo de alta de juros (2021–2023)** e o **bull market de 2024–2026** — o que enriquece muito o aprendizado do modelo, pois ele vê padrões de diferentes regimes de mercado.

> ⚠️ O arquivo deve estar na **mesma pasta** deste notebook para o carregamento funcionar corretamente.

In [ ]:
# ── Leitura do CSV ────────────────────────────────────────────────────────────
# O formato do investing.com usa ponto como separador de milhar e vírgula como decimal
# parse_dates e dayfirst garantem que as datas no formato DD.MM.YYYY sejam lidas corretamente
df_ibov = pd.read_csv(
    'IBOVESPA_historicos_012020_032026.csv',
    thousands='.',        # ponto = separador de milhar (ex: 132.697 → 132697)
    decimal=',',          # vírgula = separador decimal (ex: 1,46 → 1.46)
    parse_dates=['Data'], # converte a coluna Data para datetime automaticamente
    dayfirst=True         # interpreta datas como DD.MM.YYYY
)

# ── Renomear colunas ──────────────────────────────────────────────────────────
# Substituímos os nomes originais (com acentos) por versões simples e padronizadas
# Isso facilita o uso no código e evita erros com caracteres especiais
df_ibov.rename(columns={
    'Último':   'fechamento',   # preço de fechamento do pregão
    'Abertura': 'abertura',     # preço de abertura
    'Máxima':   'maxima',       # maior preço atingido no dia
    'Mínima':   'minima',       # menor preço atingido no dia
    'Vol.':     'volume',       # volume financeiro negociado
    'Var%':     'variacao_pct', # variação percentual em relação ao dia anterior
    'Data':     'data'          # data do pregão
}, inplace=True)

# ── Ordenação cronológica ─────────────────────────────────────────────────────
# O CSV do investing.com vem do mais recente para o mais antigo
# Para séries temporais, SEMPRE ordenar do passado para o futuro
df_ibov.sort_values('data', inplace=True)
df_ibov.reset_index(drop=True, inplace=True)  # reindexar após ordenação

# ── Conversão do Volume ───────────────────────────────────────────────────────
# O volume vem como string com sufixos: '14,37B' (bilhões) ou '850M' (milhões)
# Esta função converte para float puro (ex: 14370000000.0)
def converter_volume(v):
    if pd.isna(v) or v == '-': return np.nan   # trata valores ausentes
    v = str(v).replace(',', '.')               # substitui vírgula por ponto
    if 'B' in v: return float(v.replace('B', '')) * 1e9   # bilhões
    if 'M' in v: return float(v.replace('M', '')) * 1e6   # milhões
    if 'K' in v: return float(v.replace('K', '')) * 1e3   # milhares
    return float(v)  # número puro sem sufixo

df_ibov['volume'] = df_ibov['volume'].apply(converter_volume)

# ── Conversão da variação percentual ─────────────────────────────────────────
# A coluna Var% vem como string '1,46%' — removemos o símbolo e convertemos para float
if df_ibov['variacao_pct'].dtype == object:
    df_ibov['variacao_pct'] = (
        df_ibov['variacao_pct']
        .str.replace('%', '', regex=False)   # remove o símbolo %
        .str.replace(',', '.', regex=False)  # substitui vírgula decimal por ponto
        .astype(float)                       # converte string para número
    )

# ── Normalização da data ──────────────────────────────────────────────────────
# Garante que a data não carregue informação de horário (00:00:00)
# Isso é necessário para alinhar corretamente com os dados externos
df_ibov['data'] = pd.to_datetime(df_ibov['data']).dt.normalize()

# ── Resultado ─────────────────────────────────────────────────────────────────
print(f'IBOVESPA carregado: {len(df_ibov)} pregões')
print(f'Período: {df_ibov["data"].min().date()} → {df_ibov["data"].max().date()}')
df_ibov.head()

---
## 3. 🌐 Download dos Dados Externos (yfinance)

### Por que dados externos?

O diagnóstico das versões anteriores mostrou que o histórico de preços do IBOVESPA sozinho tem **autocorrelação do target próxima de zero** — o índice não tem "memória" detectável que prediz o dia seguinte.

Dados externos resolvem esse problema porque capturam **forças macroeconômicas globais** que movimentam o mercado brasileiro antes mesmo que ele abra:

| Ativo | Ticker | Relação esperada com IBOVESPA |
|-------|--------|-------------------------------|
| **S&P 500** | `^GSPC` | Positiva — Wall Street lidera mercados emergentes |
| **VIX** | `^VIX` | Negativa — VIX alto = pânico global → IBOV cai |
| **Dólar (USD/BRL)** | `USDBRL=X` | Negativa — dólar alto = saída de capital → IBOV cai |
| **Petróleo Brent** | `BZ=F` | Positiva — Petrobras representa ~15% do índice |

> ⚠️ **Esta célula requer conexão com a internet.** O yfinance baixa dados diretamente do Yahoo Finance de forma gratuita.

In [ ]:
# ── Parâmetros do período de download ────────────────────────────────────────
# Começamos um mês antes do IBOVESPA para garantir que médias móveis iniciais
# (como MM20 e MM50) já estejam calculadas no primeiro pregão do IBOVESPA
DATA_INICIO = '2019-12-01'
DATA_FIM    = '2026-03-20'

# ── Dicionário de tickers ─────────────────────────────────────────────────────
# Chave = ticker do Yahoo Finance | Valor = nome simplificado para uso no DataFrame
TICKERS = {
    '^GSPC'   : 'sp500',    # S&P 500 — índice das 500 maiores empresas dos EUA
    '^VIX'    : 'vix',      # Volatility Index — mede o "medo" do mercado americano
    'USDBRL=X': 'dolar',    # Taxa de câmbio Dólar americano / Real brasileiro
    'BZ=F'    : 'petroleo', # Contrato futuro do Petróleo Brent (referência global)
}

# ── Loop de download ──────────────────────────────────────────────────────────
dados_ext = {}  # dicionário para armazenar os DataFrames de cada ativo

for ticker, nome in TICKERS.items():
    print(f'Baixando {ticker} ({nome})...', end=' ')

    # yf.download retorna OHLCV (abertura, máxima, mínima, fechamento, volume)
    # auto_adjust=True já aplica ajustes por dividendos e splits automaticamente
    raw = yf.download(ticker, start=DATA_INICIO, end=DATA_FIM,
                      auto_adjust=True, progress=False)

    # Normalizar o índice para garantir que as datas não tenham horário
    raw.index = pd.to_datetime(raw.index).normalize()

    # O yfinance às vezes retorna colunas como MultiIndex (ex: ('Close', '^GSPC'))
    # Aqui achatamos para uma lista simples de strings lowercase
    if isinstance(raw.columns, pd.MultiIndex):
        raw.columns = [c[0].lower() for c in raw.columns]
    else:
        raw.columns = [c.lower() for c in raw.columns]

    # Guardamos apenas o preço de fechamento (close) e renomeamos para o nome do ativo
    dados_ext[nome] = raw[['close']].rename(columns={'close': nome})
    print(f'{len(raw)} dias ✅')

print()
print('✅ Todos os dados externos baixados!')

---
## 4. 🔗 Fusão e Alinhamento das Bases

### O desafio do alinhamento temporal

O IBOVESPA e os mercados externos **não operam nos mesmos dias**: o mercado americano tem seus próprios feriados e o câmbio tem liquidez em dias que a bolsa brasileira não opera. Isso gera lacunas quando tentamos cruzar as bases.

**Solução: `reindex` + `forward fill`**  
Usamos as **datas do IBOVESPA como âncora** e, para cada data sem cotação nos dados externos, propagamos o **último valor disponível** (`ffill`). Isso é tecnicamente correto porque:
- O mercado usa o último fechamento como referência enquanto não há nova cotação
- Não há risco de data leakage — estamos propagando informação do **passado**, não do futuro

In [ ]:
# ── Definir as datas do IBOVESPA como índice âncora ──────────────────────────
# Todo o alinhamento será feito para que cada linha corresponda a um pregão do IBOVESPA
df = df_ibov.set_index('data').copy()

# ── Juntar os dados externos com forward fill ─────────────────────────────────
for nome, serie in dados_ext.items():
    # reindex: tenta casar as datas do dado externo com as datas do IBOVESPA
    # method='ffill': para datas sem cotação, usa o último valor disponível
    alinhado = serie.reindex(df.index, method='ffill')
    df[nome] = alinhado[nome]  # adiciona a coluna ao DataFrame principal

# ── Restaurar a coluna 'data' como coluna normal ──────────────────────────────
df.reset_index(inplace=True)
df.rename(columns={'index': 'data'}, inplace=True)

# ── Verificação de qualidade ──────────────────────────────────────────────────
print('Colunas após fusão:')
print(df.columns.tolist())
print(f'\nNulos por coluna (esperado: 0 ou poucos no início do período):')
print(df[['sp500', 'vix', 'dolar', 'petroleo']].isnull().sum())
print(f'\nTotal de registros: {len(df)}')

# Visualizar as últimas linhas para confirmar o alinhamento
df[['data', 'fechamento', 'sp500', 'vix', 'dolar', 'petroleo']].tail()

In [ ]:
# ── Visualização: IBOVESPA vs cada dado externo ───────────────────────────────
# Usamos eixo duplo (twinx) para comparar séries com escalas diferentes
# O IBOVESPA fica sempre em azul claro no eixo esquerdo
# O dado externo fica colorido no eixo direito

# Filtrar linhas sem dados externos para evitar gaps nos gráficos
df_plot = df.dropna(subset=['sp500', 'vix', 'dolar', 'petroleo']).copy()

fig, axes = plt.subplots(4, 1, figsize=(16, 12), sharex=True)  # 4 subplots empilhados

# Cada par: (coluna do dado externo, rótulo do eixo, cor)
pares = [
    ('sp500',    'S&P 500',           'steelblue'),
    ('vix',      'VIX (Medo Global)', 'tomato'),
    ('dolar',    'Dólar (USD/BRL)',   'darkorange'),
    ('petroleo', 'Petróleo Brent',    'green'),
]

for ax, (col, label, cor) in zip(axes, pares):
    ax2 = ax.twinx()  # cria eixo Y direito independente para o dado externo

    # IBOVESPA no eixo esquerdo (referência)
    ax.plot(df_plot['data'], df_plot['fechamento'],
            color='lightsteelblue', linewidth=0.8, alpha=0.7, label='IBOVESPA')

    # Dado externo no eixo direito
    ax2.plot(df_plot['data'], df_plot[col],
             color=cor, linewidth=0.9, label=label)

    ax.set_ylabel('IBOVESPA (pts)', fontsize=9)
    ax2.set_ylabel(label, fontsize=9, color=cor)
    ax.set_title(f'IBOVESPA vs {label}', fontsize=11)
    ax2.tick_params(axis='y', labelcolor=cor)  # cor dos números do eixo direito

plt.tight_layout()
plt.show()

---
## 5. 🔍 Análise Exploratória — Correlações com o Target

### O que buscamos aqui?

Antes de construir o modelo, precisamos entender **se os dados externos realmente carregam sinal preditivo**. Fazemos isso calculando a correlação entre o **retorno de cada ativo HOJE** e o **target do IBOVESPA AMANHÃ**.

Se a correlação for significativamente diferente de zero, isso confirma que o dado externo ajuda a prever a direção do IBOVESPA no dia seguinte.

> 💡 **Por que isso funciona sem data leakage?**  
> O S&P500 e o dólar fecham **antes** do IBOVESPA abrir no dia seguinte. Ou seja, o retorno de hoje nesses ativos já é informação disponível quando o IBOVESPA ainda vai abrir amanhã.

In [ ]:
# ── Criação do Target ─────────────────────────────────────────────────────────
# Target = 1 se o fechamento do DIA SEGUINTE for maior que o de HOJE → ALTA
# Target = 0 se o fechamento do DIA SEGUINTE for menor ou igual → BAIXA
# shift(-1) "traz" o fechamento do próximo pregão para a linha atual
df['target'] = (df['fechamento'].shift(-1) > df['fechamento']).astype(int)

# ── Retorno diário do próprio IBOVESPA ────────────────────────────────────────
# pct_change() calcula a variação percentual em relação ao dia anterior
df['retorno'] = df['fechamento'].pct_change()

# ── Retornos diários dos dados externos ───────────────────────────────────────
# Criamos o retorno de cada ativo para usar como feature e para análise
df['ret_sp500']    = df['sp500'].pct_change()
df['ret_vix']      = df['vix'].pct_change()
df['ret_dolar']    = df['dolar'].pct_change()
df['ret_petroleo'] = df['petroleo'].pct_change()

# ── Correlação de cada ativo com o target do dia seguinte ────────────────────
# Quanto mais diferente de 0, mais o ativo contribui para prever o IBOVESPA
print('=== Correlação dos Dados Externos com o Target (IBOVESPA amanhã) ===')
print('Positivo = ativo subindo hoje aumenta probabilidade de IBOVESPA subir amanhã')
print('Negativo = ativo subindo hoje aumenta probabilidade de IBOVESPA cair amanhã')
print()

externos = ['ret_sp500', 'ret_vix', 'ret_dolar', 'ret_petroleo', 'retorno']
labels   = ['S&P500 hoje', 'VIX hoje', 'Dólar hoje', 'Petróleo hoje', 'IBOVESPA hoje']

for col, lab in zip(externos, labels):
    corr = df[col].corr(df['target'])  # correlação de Pearson entre feature e target
    print(f'  {lab:22s}: {corr:+.4f}')

print()
print('→ Dados externos têm correlações maiores que o próprio IBOVESPA!')
print('→ Isso justifica a estratégia de incorporar essas variáveis ao modelo.')

In [ ]:
# ── Mapa de calor: correlação entre todos os ativos e o target ────────────────
# O heatmap mostra de uma vez todas as correlações cruzadas
# Verde = correlação positiva | Vermelho = correlação negativa
# Os valores na diagonal são sempre 1.0 (correlação de cada série consigo mesma)

cols_corr  = ['retorno', 'ret_sp500', 'ret_vix', 'ret_dolar', 'ret_petroleo', 'target']
nomes_corr = ['Ret. IBOV', 'Ret. S&P500', 'Ret. VIX', 'Ret. Dólar', 'Ret. Petróleo', 'Target']

# Calcular a matriz de correlação apenas nas linhas sem valores ausentes
corr_matrix = df[cols_corr].dropna().corr()
corr_matrix.index   = nomes_corr
corr_matrix.columns = nomes_corr

plt.figure(figsize=(8, 6))
sns.heatmap(
    corr_matrix,
    annot=True,          # exibe os valores numéricos em cada célula
    fmt='.3f',           # formato com 3 casas decimais
    cmap='RdYlGn',       # paleta: vermelho (negativo) → amarelo (zero) → verde (positivo)
    center=0,            # ancora o centro da paleta no zero
    square=True,         # células quadradas
    linewidths=0.5,      # linhas separadoras entre células
    cbar_kws={'shrink': 0.8}
)
plt.title('Correlação entre Retornos e Target (IBOVESPA Amanhã)')
plt.tight_layout()
plt.show()

---
## 6. ⚙️ Engenharia de Atributos — IBOVESPA + Dados Externos

### Estratégia de features

Para cada dado externo, criamos um conjunto de features derivadas seguindo a mesma lógica aplicada ao IBOVESPA nas versões anteriores: retornos em diferentes janelas, médias móveis, volatilidade e indicadores de tendência.

**Regra fundamental:** todas as features usam apenas dados do **passado em relação ao ponto previsto**. Não existe nenhuma variável que "olha para o futuro".

| Grupo | Nº de features | O que captura |
|-------|---------------|---------------|
| IBOVESPA técnico | ~30 | Padrões internos do próprio índice |
| S&P 500 | 9 | Tendência do mercado americano |
| VIX | 8 | Sentimento de risco global |
| Dólar | 7 | Fluxo de capital estrangeiro |
| Petróleo | 6 | Impacto em commodities e Petrobras |
| Interações cruzadas | 3 | Sinal combinado de múltiplos mercados |

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# GRUPO 1: Features do próprio IBOVESPA
# ══════════════════════════════════════════════════════════════════════════════

# -- Retornos lagged: variação percentual em diferentes janelas ----------------
# Captura o "momentum" do índice — se ele vem subindo ou caindo
# pct_change(n) = (fechamento_hoje - fechamento_n_dias_atrás) / fechamento_n_dias_atrás
for n in [1, 2, 3, 5, 10, 15, 20]:
    df[f'ibov_ret_{n}d'] = df['fechamento'].pct_change(n)

# -- Médias Móveis e distâncias ------------------------------------------------
# A média móvel suaviza o ruído diário e revela a direção da tendência
# A distância mostra se o preço está "esticado" acima ou abaixo da tendência
for w in [5, 10, 20, 50, 200]:
    df[f'ibov_mm{w}']      = df['fechamento'].rolling(w).mean()  # média dos últimos w dias
    df[f'ibov_dist_mm{w}'] = (df['fechamento'] - df[f'ibov_mm{w}']) / df[f'ibov_mm{w}']  # distância relativa

# -- Volatilidade histórica ----------------------------------------------------
# Desvio padrão dos retornos na janela: alta volatilidade = mercado instável
for w in [5, 10, 20]:
    df[f'ibov_vol_{w}d'] = df['ibov_ret_1d'].rolling(w).std()

# -- Análise de candle (comportamento intraday) --------------------------------
# Amplitude: variação máx/mín relativa ao fechamento — indica força do movimento
df['ibov_amplitude'] = (df['maxima'] - df['minima']) / df['fechamento']

# Posição no range: onde o preço fechou dentro da variação do dia
# 1.0 = fechou na máxima (compradores dominaram) | 0.0 = fechou na mínima (vendedores dominaram)
df['ibov_posicao_range'] = (
    (df['fechamento'] - df['minima']) /
    (df['maxima'] - df['minima'] + 1e-9)  # +1e-9 evita divisão por zero
)

# Gap de abertura: diferença entre abertura de hoje e fechamento de ontem
# Gap positivo = mercado abriu otimista | Gap negativo = abriu pessimista
df['ibov_gap_abertura'] = (df['abertura'] - df['fechamento'].shift(1)) / df['fechamento'].shift(1)

# Corpo do candle: variação entre abertura e fechamento
# Positivo = dia de alta | Negativo = dia de baixa
df['ibov_corpo_candle'] = (df['fechamento'] - df['abertura']) / df['abertura']

# Sombra superior: espaço entre o topo do corpo e a máxima do dia
# Sombra longa = os compradores tentaram subir mais, mas foram rejeitados
df['ibov_sombra_sup'] = (
    (df['maxima'] - df[['fechamento', 'abertura']].max(axis=1)) / df['fechamento']
)

# Sombra inferior: espaço entre o fundo do corpo e a mínima do dia
# Sombra longa = os vendedores tentaram derrubar mais, mas foram rejeitados
df['ibov_sombra_inf'] = (
    (df[['fechamento', 'abertura']].min(axis=1) - df['minima']) / df['fechamento']
)

# -- Volume -------------------------------------------------------------------
# Volume normalizado: volume de hoje em relação à média dos últimos 10 dias
# >1.0 = volume acima do normal (possível movimento relevante)
df['ibov_vol_norm'] = df['volume'] / df['volume'].rolling(10).mean()

# -- Momentum -----------------------------------------------------------------
# Retorno acumulado relativo a um ponto passado — captura tendência de médio prazo
df['ibov_momentum_5d']  = df['fechamento'] / df['fechamento'].shift(5) - 1
df['ibov_momentum_10d'] = df['fechamento'] / df['fechamento'].shift(10) - 1

# Razão entre volatilidade de curto e longo prazo
# >1 = mercado mais agitado que o normal (regime de alta volatilidade)
df['ibov_razao_vol'] = df['ibov_vol_5d'] / (df['ibov_vol_20d'] + 1e-9)

# Aceleração do retorno: derivada do retorno — positivo = acelerando para cima
df['ibov_aceleracao'] = df['ibov_ret_1d'] - df['ibov_ret_2d']

# -- Cruzamentos de médias (sinais de tendência) ------------------------------
# Quando a MM curta cruza acima da MM longa = sinal de alta (1)
# Quando a MM curta cruza abaixo da MM longa = sinal de baixa (0)
df['ibov_regime_bull']    = (df['ibov_mm50']  > df['ibov_mm200']).astype(int)  # Golden/Death Cross
df['ibov_cross_mm5_20']   = (df['ibov_mm5']   > df['ibov_mm20']).astype(int)  # tendência de curto prazo
df['ibov_cross_mm50_200'] = (df['ibov_mm50']  > df['ibov_mm200']).astype(int) # tendência de longo prazo

# -- RSI-14 (Relative Strength Index) ----------------------------------------
# Mede a força relativa dos movimentos de alta vs baixa nos últimos 14 dias
# RSI > 70 = mercado sobrecomprado (possível reversão de baixa)
# RSI < 30 = mercado sobrevendido (possível reversão de alta)
delta         = df['fechamento'].diff()                          # variação diária
ganho         = delta.clip(lower=0).rolling(14).mean()           # média dos dias de alta
perda         = (-delta.clip(upper=0)).rolling(14).mean()        # média dos dias de baixa
df['ibov_rsi_14'] = 100 - (100 / (1 + ganho / (perda + 1e-9))) # fórmula do RSI

# Dia da semana como feature categórica (0=segunda ... 4=sexta)
# Captura possíveis efeitos de sazonalidade semanal no mercado
df['dia_semana'] = df['data'].dt.dayofweek

print('✅ Features IBOVESPA criadas.')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# GRUPO 2: Features do S&P 500
# ══════════════════════════════════════════════════════════════════════════════
# O S&P 500 fecha às 17h (horário de Brasília), antes do IBOVESPA encerrar
# Seu retorno do dia já é informação disponível quando o IBOVESPA abre no dia seguinte
# Essa é a principal vantagem desta feature: ela prediz sem causar data leakage

df['sp500_ret_1d']    = df['sp500'].pct_change(1)               # retorno do dia
df['sp500_ret_3d']    = df['sp500'].pct_change(3)               # retorno dos últimos 3 dias
df['sp500_ret_5d']    = df['sp500'].pct_change(5)               # retorno da semana
df['sp500_mm5']       = df['sp500'].rolling(5).mean()           # média de curto prazo
df['sp500_mm20']      = df['sp500'].rolling(20).mean()          # média de médio prazo
df['sp500_dist_mm20'] = (df['sp500'] - df['sp500_mm20']) / df['sp500_mm20']  # posição relativa à MM20
df['sp500_vol_5d']    = df['sp500_ret_1d'].rolling(5).std()     # volatilidade de curto prazo
df['sp500_momentum']  = df['sp500'] / df['sp500'].shift(10) - 1 # momentum de 2 semanas
df['sp500_bull']      = (df['sp500_mm5'] > df['sp500_mm20']).astype(int)  # 1 = S&P em tendência de alta

print('✅ Features S&P 500 criadas.')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# GRUPO 3: Features do VIX
# ══════════════════════════════════════════════════════════════════════════════
# O VIX mede a volatilidade implícita nas opções do S&P500 nos próximos 30 dias
# É chamado de "índice do medo": VIX alto = investidores assustados = mercados caem
#
# Faixas de referência:
#   VIX < 15  → mercado calmo, complacente
#   15 < VIX < 25 → nível normal de cautela
#   VIX > 25  → estresse elevado
#   VIX > 35  → pânico (ex: COVID março/2020 atingiu 85)

df['vix_nivel']       = df['vix']                               # nível absoluto do VIX
df['vix_ret_1d']      = df['vix'].pct_change(1)                 # variação diária do VIX
df['vix_mm5']         = df['vix'].rolling(5).mean()             # média de curto prazo
df['vix_dist_mm5']    = (df['vix'] - df['vix_mm5']) / df['vix_mm5']  # VIX acima/abaixo da média
df['vix_alto']        = (df['vix'] > 25).astype(int)            # 1 = zona de estresse
df['vix_muito_alto']  = (df['vix'] > 35).astype(int)            # 1 = zona de pânico
df['vix_baixo']       = (df['vix'] < 15).astype(int)            # 1 = mercado complacente
df['vix_variacao_5d'] = df['vix'].pct_change(5)                 # tendência do medo na semana

print('✅ Features VIX criadas.')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# GRUPO 4: Features do Dólar (USD/BRL)
# ══════════════════════════════════════════════════════════════════════════════
# Quando o dólar sobe frente ao real:
#   → Capital estrangeiro sai do Brasil (mais caro para investir em reais)
#   → Empresas exportadoras ganham (receita em dólares), mas importadoras perdem
#   → O resultado líquido para o IBOVESPA é geralmente NEGATIVO
# Essa correlação negativa dólar/IBOVESPA é uma das mais estáveis do mercado brasileiro

df['dolar_ret_1d']    = df['dolar'].pct_change(1)                # variação diária do câmbio
df['dolar_ret_5d']    = df['dolar'].pct_change(5)                # variação semanal do câmbio
df['dolar_mm5']       = df['dolar'].rolling(5).mean()            # média de curto prazo
df['dolar_mm20']      = df['dolar'].rolling(20).mean()           # média de médio prazo
df['dolar_dist_mm5']  = (df['dolar'] - df['dolar_mm5']) / df['dolar_mm5']  # posição relativa
df['dolar_tendencia'] = (df['dolar_mm5'] > df['dolar_mm20']).astype(int)   # 1 = dólar em alta (sinal negativo p/ IBOV)
df['dolar_vol_5d']    = df['dolar_ret_1d'].rolling(5).std()      # volatilidade cambial de curto prazo

print('✅ Features Dólar criadas.')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# GRUPO 5: Features do Petróleo Brent
# ══════════════════════════════════════════════════════════════════════════════
# O petróleo influencia o IBOVESPA principalmente via Petrobras (PETR3/PETR4)
# que representa sozinha ~12-15% do peso do índice
# Além disso, petróleo alto = economia global aquecida = bom para exportadoras brasileiras
# Correlação esperada: POSITIVA (petróleo sobe → IBOV sobe)

df['pet_ret_1d']    = df['petroleo'].pct_change(1)               # variação diária do petróleo
df['pet_ret_5d']    = df['petroleo'].pct_change(5)               # variação semanal
df['pet_mm5']       = df['petroleo'].rolling(5).mean()           # média de curto prazo
df['pet_mm20']      = df['petroleo'].rolling(20).mean()          # média de médio prazo
df['pet_dist_mm20'] = (df['petroleo'] - df['pet_mm20']) / df['pet_mm20']  # posição relativa à MM20
df['pet_tendencia'] = (df['pet_mm5'] > df['pet_mm20']).astype(int)        # 1 = petróleo em tendência de alta

print('✅ Features Petróleo criadas.')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# GRUPO 6: Features de interação cruzada entre mercados
# ══════════════════════════════════════════════════════════════════════════════
# Captura situações em que DOIS ativos apontam na mesma direção ao mesmo tempo
# Isso é um sinal mais robusto do que um ativo sozinho
#
# Exemplos:
#   sp500 subiu E dólar caiu hoje → sinal duplo de alta para o IBOVESPA amanhã
#   petróleo subiu E dólar caiu   → ótimo para o IBOVESPA (Petrobras + câmbio)

# S&P500 up × Dólar down: multiplica os retornos com sinal invertido no dólar
# Resultado positivo = ambos favorecem alta no IBOVESPA
df['sp500_x_dolar'] = df['sp500_ret_1d'] * (-df['dolar_ret_1d'])

# S&P500 up × VIX down: mercado americano subindo + medo caindo = risk-on global
df['sp500_x_vix']   = df['sp500_ret_1d'] * (-df['vix_ret_1d'])

# Petróleo up × Dólar down: duplo fator positivo para o IBOVESPA
df['pet_x_dolar']   = df['pet_ret_1d'] * (-df['dolar_ret_1d'])

print('✅ Features de interação cruzada criadas.')

# ── Lista final de features ───────────────────────────────────────────────────
# Excluímos colunas brutas (preços, volumes originais) e médias intermediárias
# que foram usadas para calcular as features mas não devem entrar no modelo diretamente
EXCLUIR = [
    'data', 'fechamento', 'abertura', 'maxima', 'minima', 'volume',
    'variacao_pct', 'target', 'retorno',
    'sp500', 'vix', 'dolar', 'petroleo',          # preços brutos externos
    'ret_sp500', 'ret_vix', 'ret_dolar', 'ret_petroleo',  # retornos da EDA (já nas features)
    'ibov_mm5', 'ibov_mm10', 'ibov_mm20', 'ibov_mm50', 'ibov_mm200',  # MMs intermediárias
    'sp500_mm5', 'sp500_mm20',   # MMs do S&P500 (usadas para calcular as features)
    'dolar_mm5', 'dolar_mm20',   # MMs do dólar
    'pet_mm5', 'pet_mm20',       # MMs do petróleo
    'vix_mm5',                   # MM do VIX
]

FEATURES = [c for c in df.columns if c not in EXCLUIR]

print(f'\n✅ Total de features para o modelo: {len(FEATURES)}')

# Resumo por grupo
grupos = {}
for f in FEATURES:
    prefix = f.split('_')[0]
    grupos.setdefault(prefix, []).append(f)
print('\nFeatures por grupo:')
for g, fs in sorted(grupos.items()):
    print(f'  {g:10s}: {len(fs):2d} features')

---
## 7. 🗂️ Divisão Treino/Teste e Validação Cruzada Temporal

### Regras inegociáveis para séries temporais

Em séries temporais, qualquer mistura temporal nos dados contamina o modelo com informação do futuro — o que chamamos de **data leakage**. Seguimos três proteções obrigatórias:

1. **Nunca embaralhar** — a ordem cronológica é preservada em todos os momentos
2. **Teste = últimos 30 dias** — exigência do briefing e boa prática de mercado
3. **Scaler ajustado apenas no treino** — a escala do período de teste não pode influenciar a normalização

### O que é o TimeSeriesSplit?

O `TimeSeriesSplit` é a versão temporal do cross-validation clássico. Em vez de sortear folds aleatoriamente (o que misturaria passado e futuro), ele cria janelas expansivas: cada fold treina em dados mais antigos e testa em dados mais recentes, simulando como o modelo performaria em diferentes momentos do passado.

In [ ]:
# ── Limpeza: remover linhas com valores ausentes ──────────────────────────────
# As janelas deslizantes (rolling) geram NaN nas primeiras N linhas de cada feature
# A última linha tem NaN no target (não há 'amanhã' para prever)
# dropna() remove todas essas linhas incompletas
df_model = df[FEATURES + ['target', 'data']].dropna().copy()

# ── Divisão temporal: treino e teste ─────────────────────────────────────────
DIAS_TESTE = 30  # últimos 30 pregões vão para o conjunto de teste

df_treino = df_model.iloc[:-DIAS_TESTE]  # todos exceto os últimos 30 dias
df_teste  = df_model.iloc[-DIAS_TESTE:]  # exatamente os últimos 30 dias

# Separar features (X) e target (y) para treino e teste
X_treino, y_treino = df_treino[FEATURES], df_treino['target']
X_teste,  y_teste  = df_teste[FEATURES],  df_teste['target']

# ── Normalização (StandardScaler) ────────────────────────────────────────────
# Transforma cada feature para média 0 e desvio padrão 1
# Necessário para a Regressão Logística, que é sensível à escala
# CRÍTICO: fit_transform() APENAS no treino → não vazar escala do teste!
scaler      = StandardScaler()
X_treino_sc = scaler.fit_transform(X_treino)  # aprende a escala E normaliza
X_teste_sc  = scaler.transform(X_teste)       # aplica a MESMA escala do treino

# ── Resumo da divisão ─────────────────────────────────────────────────────────
print(f'Treino : {len(df_treino):>5} dias | {df_treino["data"].min().date()} → {df_treino["data"].max().date()}')
print(f'Teste  : {len(df_teste):>5} dias | {df_teste["data"].min().date()}  → {df_teste["data"].max().date()}')
print(f'Features totais: {len(FEATURES)}')
print(f'\nBalanceamento do target no teste:')
print(f'  Alta  (1): {y_teste.sum()} dias ({y_teste.mean()*100:.1f}%)')
print(f'  Baixa (0): {(y_teste==0).sum()} dias ({(1-y_teste.mean())*100:.1f}%)')

In [ ]:
# ── Cross-Validation Temporal (TimeSeriesSplit, 5 folds) ──────────────────────
# Cada fold segue a lógica: treina em dados anteriores, testa nos seguintes
# Isso simula como o modelo teria performado em diferentes momentos do passado
# e fornece uma estimativa mais honesta da acurácia real

tscv = TimeSeriesSplit(n_splits=5)  # 5 folds temporais

print('=== Cross-Validation Temporal (5 folds) — COM dados externos ===')
print('Referência (versão anterior sem dados externos): CV ~51% ±4%')
print()

# Definir os modelos com seus dados (escalados ou não)
modelos_cv = [
    # (nome, instância do modelo, dados a usar)
    ('Regressão Logística',
     LogisticRegression(max_iter=2000, C=0.05, random_state=42),
     X_treino_sc),  # Logística precisa de dados normalizados

    ('Random Forest',
     RandomForestClassifier(n_estimators=500, max_depth=6, min_samples_leaf=8,
                            max_features='sqrt', random_state=42, n_jobs=-1),
     X_treino),     # RF não precisa de normalização

    ('Gradient Boosting',
     GradientBoostingClassifier(n_estimators=500, max_depth=4,
                                learning_rate=0.01, subsample=0.85, random_state=42),
     X_treino),     # GB não precisa de normalização
]

for nome, modelo, X in modelos_cv:
    # cross_val_score executa o CV e retorna um score por fold
    scores = cross_val_score(modelo, X, y_treino, cv=tscv, scoring='accuracy')
    folds  = [f'{s*100:.0f}%' for s in scores]  # formatar para exibição
    print(f'{nome:25s}: CV={scores.mean()*100:.1f}% ±{scores.std()*100:.1f}%  →  folds={folds}')

print()
print('→ Comparar com referência para quantificar o ganho dos dados externos.')

---
## 8. 📊 Modelo 1: Regressão Logística

### Papel no projeto: Baseline interpretável

A Regressão Logística é o nosso **ponto de partida**. Ela modela a probabilidade de alta como uma combinação linear ponderada das features. Apesar de simples, é valiosa por duas razões:

- **Interpretabilidade:** os coeficientes revelam quais features têm relação estatisticamente significativa com a direção do IBOVESPA, com sinal positivo (favorece alta) ou negativo (favorece baixa)
- **Sanity check:** se um modelo mais complexo não superar a Regressão Logística, significa que a complexidade adicional não está valendo a pena

**Hiperparâmetro-chave:** `C=0.05` — controla a regularização L2. Quanto menor o C, mais forte a penalização de coeficientes grandes, reduzindo o risco de overfitting.

In [ ]:
# ── Treinamento ───────────────────────────────────────────────────────────────
# C=0.05: regularização L2 forte — protege contra overfitting com muitas features
# max_iter=2000: número máximo de iterações do otimizador para garantir convergência
# random_state=42: garante reprodutibilidade dos resultados
lr_model = LogisticRegression(max_iter=2000, C=0.05, random_state=42)
lr_model.fit(X_treino_sc, y_treino)  # treinar com dados normalizados

# ── Predição no conjunto de teste ─────────────────────────────────────────────
# predict() retorna a classe prevista (0 ou 1) para cada dia do teste
y_pred_lr   = lr_model.predict(X_teste_sc)
acuracia_lr = accuracy_score(y_teste, y_pred_lr)  # % de acertos

print(f'Acurácia — Regressão Logística: {acuracia_lr*100:.2f}%')
print()

# classification_report mostra métricas separadas para cada classe:
# precision: dos que previmos como Alta, quantos % realmente foram Alta?
# recall: de todos os dias que foram Alta, quantos % o modelo acertou?
# f1-score: média harmônica entre precision e recall
print(classification_report(y_teste, y_pred_lr, target_names=['Baixa', 'Alta']))

In [ ]:
# ── Visualização: Matriz de Confusão + Coeficientes ──────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# -- Matriz de confusão --------------------------------------------------------
# Linhas = classe real | Colunas = classe prevista
# Diagonal principal (canto sup-esq e inf-dir) = acertos
# Fora da diagonal = erros: falso positivo e falso negativo
ConfusionMatrixDisplay(
    confusion_matrix(y_teste, y_pred_lr),
    display_labels=['Baixa', 'Alta']
).plot(ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title('Matriz de Confusão — Regressão Logística')

# -- Coeficientes: interpretando o modelo ------------------------------------
# Coeficiente positivo = feature aumenta probabilidade de ALTA
# Coeficiente negativo = feature aumenta probabilidade de BAIXA
# Exibimos apenas o top 15 por magnitude absoluta para não poluir o gráfico
coefs = pd.Series(lr_model.coef_[0], index=FEATURES)
top15 = coefs.reindex(coefs.abs().sort_values(ascending=False).index[:15]).sort_values()

cores_coef = ['steelblue' if c > 0 else 'tomato' for c in top15]
top15.plot(kind='barh', ax=axes[1], color=cores_coef, edgecolor='white')
axes[1].axvline(0, color='black', linewidth=0.8)  # linha de referência no zero
axes[1].set_title('Top 15 Coeficientes (azul = favorece Alta)')
axes[1].set_xlabel('Peso do coeficiente')

plt.tight_layout()
plt.show()

---
## 9. 🌲 Modelo 2: Random Forest

### Como funciona o Random Forest?

O Random Forest é um **ensemble de árvores de decisão** treinadas em paralelo. Cada árvore:
1. Recebe uma **amostra aleatória** dos dados de treino (com reposição — bootstrap)
2. Em cada nó da árvore, considera apenas uma **seleção aleatória de features** (`max_features='sqrt'`)
3. Cresce até uma profundidade máxima (`max_depth`) ou até ter poucos dados por folha

A predição final é feita por **votação majoritária** entre todas as 500 árvores. Essa diversidade reduz o overfitting que uma árvore única inevitavelmente teria.

**Por que funciona bem aqui?** O Random Forest captura **relações não-lineares** entre features — por exemplo, que o VIX só impacta o IBOVESPA de forma diferente quando o dólar também está em alta. A Regressão Logística não consegue capturar essa interação sem que ela seja criada manualmente.

In [ ]:
# ── Treinamento ───────────────────────────────────────────────────────────────
rf_model = RandomForestClassifier(
    n_estimators=500,       # número de árvores — mais = mais estável, mas mais lento
    max_depth=6,            # profundidade máxima de cada árvore — limita complexidade
    min_samples_leaf=8,     # mínimo de amostras em cada folha — evita overfitting
    max_features='sqrt',    # usa √n_features por nó — aumenta diversidade entre árvores
    random_state=42,        # reprodutibilidade
    n_jobs=-1               # usa todos os núcleos disponíveis para treinar em paralelo
)
# Random Forest não precisa de normalização — árvores são invariantes à escala
rf_model.fit(X_treino, y_treino)

# ── Predição e avaliação ──────────────────────────────────────────────────────
y_pred_rf   = rf_model.predict(X_teste)
acuracia_rf = accuracy_score(y_teste, y_pred_rf)

print(f'Acurácia — Random Forest: {acuracia_rf*100:.2f}%')
print()
print(classification_report(y_teste, y_pred_rf, target_names=['Baixa', 'Alta']))

In [ ]:
# ── Visualização: Matriz de Confusão + Importância das Features ───────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# -- Matriz de confusão --------------------------------------------------------
ConfusionMatrixDisplay(
    confusion_matrix(y_teste, y_pred_rf),
    display_labels=['Baixa', 'Alta']
).plot(ax=axes[0], colorbar=False, cmap='Greens')
axes[0].set_title('Matriz de Confusão — Random Forest')

# -- Importância das features --------------------------------------------------
# O Random Forest calcula a importância de cada feature como a média da redução
# de impureza (Gini) que ela provoca em todos os nós de todas as árvores
# Features mais altas = mais discriminativas para separar Alta de Baixa
imp_rf = pd.Series(rf_model.feature_importances_, index=FEATURES)
imp_rf.sort_values(ascending=False).head(15).sort_values().plot(
    kind='barh', ax=axes[1], color='steelblue', edgecolor='white'
)
axes[1].set_title('Top 15 Features mais Importantes — Random Forest')
axes[1].set_xlabel('Importância relativa')

plt.tight_layout()
plt.show()

print('Top 10 features por importância:')
print(imp_rf.sort_values(ascending=False).head(10).round(4))

---
## 10. ⚡ Modelo 3: Gradient Boosting

### Como difere do Random Forest?

Enquanto o Random Forest treina centenas de árvores **em paralelo e de forma independente**, o Gradient Boosting as treina **em sequência**: cada nova árvore foca nos erros cometidos pela anterior, corrigindo-os progressivamente.

É como aprender com os próprios erros a cada rodada — por isso tende a ter performance superior ao Random Forest, mas também é mais sensível a overfitting se mal configurado.

**Hiperparâmetros críticos para controlar overfitting:**
- `learning_rate=0.01` — passo muito pequeno, aprendizado conservador
- `subsample=0.85` — cada árvore vê apenas 85% dos dados (Stochastic GB), aumentando a robustez
- `n_estimators=500` — mais árvores compensam o passo pequeno do learning rate

In [ ]:
# ── Treinamento ───────────────────────────────────────────────────────────────
gb_model = GradientBoostingClassifier(
    n_estimators=500,     # número de árvores sequenciais (rounds de boosting)
    max_depth=4,          # cada árvore é rasa — potência vem da sequência, não da profundidade
    learning_rate=0.01,   # passo de correção por árvore — pequeno = mais estável
    subsample=0.85,       # Stochastic GB: usa 85% dos dados por árvore (reduz overfitting)
    random_state=42       # reprodutibilidade
)
# GB também não precisa de normalização
gb_model.fit(X_treino, y_treino)

# ── Predição e avaliação ──────────────────────────────────────────────────────
y_pred_gb   = gb_model.predict(X_teste)
acuracia_gb = accuracy_score(y_teste, y_pred_gb)

print(f'Acurácia — Gradient Boosting: {acuracia_gb*100:.2f}%')
print()
print(classification_report(y_teste, y_pred_gb, target_names=['Baixa', 'Alta']))

In [ ]:
# ── Visualização: Matriz de Confusão + Importância das Features ───────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# -- Matriz de confusão --------------------------------------------------------
ConfusionMatrixDisplay(
    confusion_matrix(y_teste, y_pred_gb),
    display_labels=['Baixa', 'Alta']
).plot(ax=axes[0], colorbar=False, cmap='Oranges')
axes[0].set_title('Matriz de Confusão — Gradient Boosting')

# -- Importância das features --------------------------------------------------
# No Gradient Boosting, a importância é medida pela frequência com que
# cada feature é escolhida para fazer divisões nas árvores, ponderada pela melhoria
imp_gb = pd.Series(gb_model.feature_importances_, index=FEATURES)
imp_gb.sort_values(ascending=False).head(15).sort_values().plot(
    kind='barh', ax=axes[1], color='darkorange', edgecolor='white'
)
axes[1].set_title('Top 15 Features mais Importantes — Gradient Boosting')
axes[1].set_xlabel('Importância relativa')

plt.tight_layout()
plt.show()

print('Top 10 features por importância:')
print(imp_gb.sort_values(ascending=False).head(10).round(4))

---
## 11. 🏆 Comparação, Ensemble e Escolha Final

### Soft Voting Ensemble — por que combinar os modelos?

Cada modelo tem seus pontos fortes e pontos cegos:
- A **Regressão Logística** é estável e captura relações lineares claras
- O **Random Forest** é robusto a outliers e captura interações complexas
- O **Gradient Boosting** é preciso e aprende padrões sutis

Quando combinamos suas **probabilidades** (soft voting), os erros de um tendem a ser compensados pelos acertos dos outros. O resultado final é mais estável e confiável do que qualquer modelo individualmente.

In [ ]:
# ── Soft Voting Ensemble ──────────────────────────────────────────────────────
# predict_proba() retorna a probabilidade de cada classe [P(Baixa), P(Alta)]
# Pegamos apenas P(Alta) — a probabilidade de o IBOVESPA subir amanhã
proba_lr = lr_model.predict_proba(X_teste_sc)[:, 1]  # coluna 1 = P(Alta)
proba_rf = rf_model.predict_proba(X_teste)[:, 1]
proba_gb = gb_model.predict_proba(X_teste)[:, 1]

# Média simples das probabilidades dos 3 modelos
proba_ens  = (proba_lr + proba_rf + proba_gb) / 3

# Decisão final: se a probabilidade média >= 0.50, prevê Alta (1), senão Baixa (0)
y_pred_ens   = (proba_ens >= 0.50).astype(int)
acuracia_ens = accuracy_score(y_teste, y_pred_ens)

# ── Tabela comparativa ────────────────────────────────────────────────────────
resultados = pd.DataFrame({
    'Modelo': ['Reg. Logística', 'Random Forest', 'Gradient Boosting', 'Soft Ensemble'],
    'Acurácia (%)': [
        round(acuracia_lr  * 100, 2),
        round(acuracia_rf  * 100, 2),
        round(acuracia_gb  * 100, 2),
        round(acuracia_ens * 100, 2),
    ],
    'Meta ≥75%': [
        '✅' if a >= 0.75 else '❌'
        for a in [acuracia_lr, acuracia_rf, acuracia_gb, acuracia_ens]
    ]
})
print(resultados.to_string(index=False))

# ── Gráfico comparativo ───────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 4))
cores = ['steelblue', 'green', 'darkorange', 'purple']
bars  = ax.bar(resultados['Modelo'], resultados['Acurácia (%)'],
               color=cores, edgecolor='white', width=0.55)

# Linha da meta (75%) e do melhor resultado anterior (60% na V1, sem dados externos)
ax.axhline(75, color='red',  linestyle='--', linewidth=1.5, label='Meta: 75%')
ax.axhline(60, color='gray', linestyle=':',  linewidth=1,
           label='Melhor anterior V1 (sem dados externos): 60%')

ax.set_ylim(0, 100)
ax.set_ylabel('Acurácia (%)')
ax.set_title('Comparação de Acurácia — COM Dados Externos (últimos 30 dias)')
ax.legend(fontsize=9)

# Anotar valor em cima de cada barra
for bar in bars:
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.8,
            f'{bar.get_height():.1f}%', ha='center', va='bottom', fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
# ── Seleção automática do modelo com maior acurácia ───────────────────────────
todos = {
    'Regressão Logística' : (acuracia_lr,  y_pred_lr),
    'Random Forest'       : (acuracia_rf,  y_pred_rf),
    'Gradient Boosting'   : (acuracia_gb,  y_pred_gb),
    'Soft Voting Ensemble': (acuracia_ens, y_pred_ens),
}
# Seleciona o modelo com maior acurácia no conjunto de teste
melhor_nome     = max(todos, key=lambda k: todos[k][0])
melhor_acuracia = todos[melhor_nome][0]
y_pred_final    = todos[melhor_nome][1]

print(f'🏆 Modelo selecionado : {melhor_nome}')
print(f'   Acurácia no teste  : {melhor_acuracia*100:.2f}%')
print()
print(classification_report(y_teste, y_pred_final, target_names=['Baixa', 'Alta']))

# ── Gráfico: previsões vs real dia a dia ──────────────────────────────────────
# Cada ponto representa um dia do período de teste
# Verde = modelo acertou | Vermelho = modelo errou
df_res = df_teste[['data']].copy()
df_res['real']     = y_teste.values
df_res['previsto'] = y_pred_final
df_res['acerto']   = (df_res['real'] == df_res['previsto'])

fig, ax = plt.subplots(figsize=(16, 3))
cores_p = df_res['acerto'].map({True: 'green', False: 'red'})

# Pontos coloridos = resultado real de cada dia
ax.scatter(df_res['data'], df_res['real'],
           c=cores_p, s=90, zorder=5, label='Real (verde=acerto, vermelho=erro)')

# Linha em degraus = o que o modelo previu
ax.step(df_res['data'], df_res['previsto'],
        color='steelblue', linewidth=1, alpha=0.7, where='mid', label='Previsto')

ax.set_title(f'Previsões vs Real — {melhor_nome} (últimos 30 dias)')
ax.set_yticks([0, 1])
ax.set_yticklabels(['Baixa', 'Alta'])
ax.legend()
plt.tight_layout()
plt.show()

acertos = df_res['acerto'].sum()
print(f'Acertos: {acertos} de 30 dias ({acertos/30*100:.1f}%)')

---
## 12. 📝 Justificativa Técnica

### Por que incluir dados externos?

O diagnóstico das versões anteriores revelou que o histórico do próprio IBOVESPA tem **autocorrelação do target próxima de zero** em todos os lags analisados. Isso indica que o índice não carrega memória preditiva detectável por esses modelos.

Dados externos resolvem esse problema porque capturam **forças externas** que o IBOVESPA ainda não "absorveu" no momento da previsão:

| Dado | Mecanismo de influência | Direção esperada |
|------|------------------------|------------------|
| S&P 500 | Bolsas emergentes seguem Wall Street; o S&P fecha antes do IBOV abrir | Positiva |
| VIX | Risco global afeta apetite por ativos emergentes | Negativa |
| Dólar | USD alto = saída de capital estrangeiro do Brasil | Negativa |
| Petróleo | Petrobras representa ~13% do índice; commodities refletem saúde global | Positiva |

### Como tratamos a natureza sequencial dos dados

1. **Alinhamento com forward fill:** dias sem cotação nos EUA/commodities recebem o último valor disponível — tecnicamente correto, pois o mercado usa esse valor como referência
2. **Todos os retornos externos usam dados do DIA ATUAL** para prever o IBOVESPA do DIA SEGUINTE — não há data leakage
3. **Divisão temporal estrita:** treino precede o teste; nenhum dado futuro contamina o treino
4. **Scaler ajustado apenas no treino:** as estatísticas de normalização (média e desvio) são aprendidas somente nos dados de treino
5. **TimeSeriesSplit para cross-validation:** 5 folds temporais em vez de k-fold aleatório

### Escolha e comparação dos modelos

| Critério | Reg. Logística | Random Forest | Gradient Boosting |
|----------|---------------|---------------|-------------------|
| Papel | Baseline linear | Ensemble paralelo | Ensemble sequencial |
| Interpretabilidade | ⭐⭐⭐ Alta | ⭐⭐ Média | ⭐ Baixa |
| Captura não-linearidades | ❌ Não | ✅ Sim | ✅ Sim |
| Risco de overfitting | 🟢 Baixo | 🟡 Médio | 🟡 Médio |
| Regularização utilizada | `C=0.05` | `max_depth=6`, `min_samples_leaf=8` | `lr=0.01`, `subsample=0.85` |
| Necessita normalização | ✅ Sim | ❌ Não | ❌ Não |

### Trade-offs Acurácia vs Overfitting

> 💡 Em séries financeiras, overfitting é o maior risco. Um modelo com CV=55% e teste=75% é infinitamente mais útil do que um com CV=95% e teste=40%. Todas as escolhas de hiperparâmetros foram orientadas para **maximizar a generalização**, não a performance no treino.

### Evolução ao longo do projeto

**Resultados detalhados por versão no conjunto de teste (últimos 30 dias):**

| Versão | Período | Registros | Features | Log. Reg. | Random Forest | XGBoost/GB | Melhor |
|--------|---------|-----------|----------|-----------|---------------|------------|--------|
| V1 | Jan/2024–Mar/2026 | 524 | 12 | 40,00% | **60,00%** | 50,00% | 60,00% (RF) |
| V2 | Jan/2020–Mar/2026 | 1.529 | 12 | 40,00% | **53,33%** | 50,00% | 53,33% (RF) |
| **V3** | **Jan/2020–Mar/2026** | **~1.300+** | **60+** | **ver acima** | **ver acima** | **ver acima** | **ver acima** |

> 💡 **Observação importante:** a V2 teve desempenho inferior à V1 mesmo com mais dados. Isso confirma que o problema não é quantidade de dados — é que o histórico do próprio índice não carrega sinal suficiente para previsão de direção diária. A solução é adicionar **informação externa**, o que é feito nesta V3.

---
## 13. 🎯 Conclusão

In [ ]:
# ── Resumo final automático ───────────────────────────────────────────────────
# Consolida todos os resultados em um único bloco de texto estruturado
print('=' * 68)
print('     RESUMO FINAL — TECH CHALLENGE FASE 02 (v3 — Dados Externos)')
print('=' * 68)
print()
print(f'  Base IBOVESPA : IBOVESPA_historicos_012020_032026.csv')
print(f'  Dados externos: S&P500 (^GSPC), VIX (^VIX), Dólar (USDBRL=X), Petróleo (BZ=F)')
print(f'  Período       : {df_model["data"].min().date()} → {df_model["data"].max().date()}')
print(f'  Registros     : {len(df_model)} pregões | Features: {len(FEATURES)}')
print()
print('  Resultados no conjunto de teste (últimos 30 dias):')
print(f'    • Regressão Logística  : {acuracia_lr*100:.2f}%')
print(f'    • Random Forest        : {acuracia_rf*100:.2f}%')
print(f'    • Gradient Boosting    : {acuracia_gb*100:.2f}%')
print(f'    • Soft Voting Ensemble : {acuracia_ens*100:.2f}%')
print()
print(f'  🏆 Modelo selecionado   : {melhor_nome}')
print(f'  📊 Acurácia final       : {melhor_acuracia*100:.2f}%')
# Status da meta
status = '✅ META ATINGIDA (≥ 75%)' if melhor_acuracia >= 0.75 else '❌ Abaixo da meta'
print(f'  🎯 Status               : {status}')
print()
print('  Evolução ao longo do projeto:')
print('    V1 | Jan/2024-Mar/2026 | 12 features | 60,00% (Random Forest)')
print('    V2 | Jan/2020-Mar/2026 | 12 features | 53,33% (Random Forest)')
print(f'    V3 | Jan/2020-Mar/2026 | {len(FEATURES)} features  | {melhor_acuracia*100:.2f}% ({melhor_nome})')
print()
print('=' * 68)

---
*Tech Challenge Fase 02 — Pós-graduação Data Analytics POSTECH*